
<table style="width: 100%; border-collapse: collapse; border-bottom: 4px solid #003366;">
    <tr style="border: none;">
        <td style="border: none; text-align: left; vertical-align: middle; padding-bottom: 15px;">
            <h1 style="color: #003366; font-family: 'Helvetica', sans-serif; margin-top: 0; margin-bottom: 5px;">IMT3410: M&eacute;todos para Ecuaciones Diferenciales</h1>
            <h2 style="color: #555; margin-top: 10px; margin-bottom: 10px;">Cap&iacute;tulo 2: Ecuaciones Diferenciales Parciales El&iacute;ticas - M&eacute;todo de Elementos Finitos para problemas elipticos lineales de segundo orden. Conformes y No conformes</h2>
            <p style="margin: 2px 0;"><b>Profesor:</b> Manuel A. Sánchez </p>
            <p style="margin: 2px 0;"><b>Institución:</b> Instituto de Ingenier&iacute'a Matemática y Coputacional, Pontificia Universidad Católica de Chile</p>
        </td>
        <td style="border: none; text-align: right; vertical-align: middle; padding-bottom: 15px; width: 260px;">
            <!-- <img src="../source/FacMatematicas-15.png" alt="Logo UC" width="400" style="margin-bottom: 8px;"> -->
            <br>
            <a href="https://colab.research.google.com/github/ManuelSanchezUribe/ManuelSanchezUribe.github.io/blob/main/IMT3410/Notebooks_2026/02_Capitulo_FEM_H1_CR.ipynb" target="_blank">
                <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" width="120">
            </a>
        </td>
    </tr>
</table>
<br>


## Capitulo 2: NGsolve FEM for elliptic problem

Sea $\Omega$ un dominio en $\mathbb R^{d}$. Considere el operador diferencial $L$
$$
L u = - \nabla\cdot (\sigma \nabla u) + \beta\cdot\nabla u + \mu u
$$
donde $\sigma:\Omega \to \mathbb R^{d,d}$, $\beta:\Omega\to \mathbb R^{d}$, $\mu\in \mathbb R$. Dada una funci\'on $f:\Omega \to \mathbb R$ y considere el problema
de hallar una funci\'on $u:\Omega\to \mathbb R$ tal que
$$
L u =f, \quad \text{en }\Omega,\qquad \mathcal B u =g,\quad \text{sobre }\partial \Omega.
$$

### Formulación Variacional (Débil)
Para implementar este problema utilizando el Método de Elementos Finitos (FEM), multiplicamos por una función de prueba $v \in V = H^1_0(\Omega)$ e integramos por partes el término difusivo. El problema se traduce en hallar $u \in V$ tal que:
$$
a(u,v) = \ell(v) \quad \forall v \in V
$$
donde la forma bilineal $a(\cdot,\cdot)$ y la forma lineal $\ell(\cdot)$ están dadas por:
$$
a(u,v) = \int_{\Omega} \left( \sigma \nabla u \cdot \nabla v + (\beta \cdot \nabla u) v + \mu u v \right) dx
$$
$$
\ell(v) = \int_{\Omega} f v \, dx
$$
En NGSolve, estas expresiones matemáticas se traducen de forma casi literal usando los objetos `BilinearForm` y `LinearForm`.

In [27]:
# Ejecutar esta celda para instalar NGSolve en Google Colab
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-real.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

In [14]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import pandas as pd

## $H^1$ solver

In [38]:
def H1_ellipticSolver(mesh, order, sigma, beta, mu, exact_sol=None, source_term=None):
    """
    Resuelve el problema elíptico usando elementos finitos H1 conformes.
    """
    Vh = H1(mesh, order=order, dirichlet='left|bottom|right|top')
    u = Vh.TrialFunction()
    v = Vh.TestFunction()

    f = LinearForm(Vh)
    # f += (sigma*2*pi**2*sin(pi*x)*sin(pi*y) +
    #       beta[0]*pi*cos(pi*x)*sin(pi*y)+beta[1]*pi*sin(pi*x)*cos(pi*y)+
    #       mu*sin(pi*x)*sin(pi*y)) *v*dx
    if source_term:
        f += source_term * v * dx

    a = BilinearForm(Vh)
    a += sigma*grad(u)*grad(v)*dx
    a += InnerProduct(grad(u),beta* v)*dx
    a += mu*u*v*dx

    a.Assemble();f.Assemble()

    gfu = GridFunction(Vh)
    gfu.vec.data = a.mat.Inverse(Vh.FreeDofs()) * f.vec

    return gfu

In [40]:
sigma = 0.1
beta = CF((1.0, 1.0))
mu = 0.5
order=1
mesh = Mesh(unit_square.GenerateMesh(maxh=0.25))
# Término fuente deducido de uexact analíticamente
f_source = (sigma * 2 * pi**2 * sin(pi*x) * sin(pi*y) +
                beta[0] * pi * cos(pi*x) * sin(pi*y) + 
                beta[1] * pi * sin(pi*x) * cos(pi*y) +
                mu * sin(pi*x) * sin(pi*y))
gfu = H1_ellipticSolver(mesh, order, sigma, beta, mu, source_term=f_source)
Draw(gfu)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene

In [67]:
def computehmax(mesh):
    elvol = Integrate(CoefficientFunction(1), mesh, element_wise=True)
    hh = [(2*vol)**(1/2) for vol in elvol]
    return max(hh)

def computerate(h, error):
    rate = np.zeros(h.size)
    for j in range(h.size-1):
        rate[j+1] = np.log(error[j+1]/error[j])/np.log(h[j+1]/h[j])
    return rate

def Convergence_Test(h0=0.25, nmeshes=6, order=1, metodo='H1'):
    # Solución manufacturada (Manufactured Solution)
    uexact = sin(pi*x)*sin(pi*y)
    
    # Parámetros del problema
    sigma = 0.1
    beta = CF((1.0, 1.0))
    mu = 0.5
    
    # Término fuente deducido de uexact analíticamente
    f_source = (sigma * 2 * pi**2 * sin(pi*x) * sin(pi*y) +
                beta[0] * pi * cos(pi*x) * sin(pi*y) + 
                beta[1] * pi * sin(pi*x) * cos(pi*y) +
                mu * sin(pi*x) * sin(pi*y))
    
    f_source_CR = (1.0 * 2 * pi**2 * sin(pi*x) * sin(pi*y))

    err_u = np.zeros(nmeshes)
    h = np.zeros(nmeshes)
    Ndof = []

    mesh = Mesh(unit_square.GenerateMesh(maxh=h0))
    
    for j in range(nmeshes):
        h[j] = computehmax(mesh)
        
        if metodo == 'CR':
            gfu = CR_ellipticSolver(mesh, order, source_term=f_source_CR)
        else: 
            gfu = H1_ellipticSolver(mesh, order, sigma, beta, mu, source_term=f_source)
            
        err_u[j] = sqrt(Integrate(InnerProduct(gfu - uexact, gfu - uexact), mesh))
        Ndof.append(gfu.space.ndof)
        
        if j < nmeshes - 1:
            mesh.Refine()
            
    rate = computerate(h, err_u)
    return h, err_u, rate, gfu, Ndof

In [74]:
# Test H1 Order 1, 2, 3
h1, err_u1, rate1, gfu1, Ndof1 = Convergence_Test(h0 = 0.25, nmeshes=6, order=1)
h2, err_u2, rate2, gfu2, Ndof2 = Convergence_Test(h0 = 0.25, nmeshes=6, order=2)
h3, err_u3, rate3, gfu3, Ndof3 = Convergence_Test(h0 = 0.25, nmeshes=6, order=3)

df_H1 = pd.DataFrame({'h':h1, 'Ndof k=1':Ndof1, 'error k=1':err_u1, 'order k=1':rate1,
                  'Ndof k=2':Ndof2,'error k=2':err_u2, 'order k=2':rate2,
                  'Ndof k=3':Ndof3,'error k=3':err_u3, 'order k=3':rate3})
df_H1


,h,Ndof k=1,error k=1,order k=1,Ndof k=2,error k=2,order k=2,Ndof k=3,error k=3,order k=3
0,0.306736,26,0.058905,0.000000,85,3.706092e-03,0.000000,178,2.961737e-04,0.000000
1,0.153368,85,0.014738,1.998813,305,9.139712e-04,2.019678,661,2.567865e-05,3.527802
2,0.076684,305,0.003633,2.020183,1153,1.180343e-04,2.952943,2545,1.649792e-06,3.960213
3,0.038342,1153,0.000901,2.012055,4481,1.501218e-05,2.975000,9985,1.026949e-07,4.005848
4,0.019171,4481,0.000224,2.005455,17665,1.892500e-06,2.987769,39553,6.378081e-09,4.009098
5,0.009586,17665,0.000056,2.002540,70145,2.375376e-07,2.994065,157441,3.969659e-10,4.006035


## Crouzeix-Raviart Solver Non conforming methods

In [66]:
def CR_ellipticSolver(mesh, order, source_term):
    sigma = 1.0

    Vh = FESpace("nonconforming", mesh,order=order, dirichlet='left|bottom|right|top')
    u = Vh.TrialFunction()
    v = Vh.TestFunction()

    f = LinearForm(Vh)

    f += (sigma*2*pi**2*sin(pi*x)*sin(pi*y) ) *v*dx


    a = BilinearForm(Vh)
    a += grad(u)*grad(v)*dx

    a.Assemble();f.Assemble()

    gfu = GridFunction(Vh)
    gfu.vec.data = a.mat.Inverse(Vh.FreeDofs()) * f.vec

    return gfu


In [58]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
f_source_CR = (1.0 * 2 * pi**2 * sin(pi*x) * sin(pi*y))
gfu = CR_ellipticSolver(mesh, f_source_CR)
Draw(gfu)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene

In [73]:

# Test CR
h1_CR, err_uCR1, rateCR1, gfuCR1, NdofCR1 = Convergence_Test(h0 = 0.25, nmeshes=5, order=1, metodo='CR')
df_CR = pd.DataFrame({'h':h1_CR, 'Ndof':NdofCR1, 'error k=1':err_uCR1, 'order k=1':rateCR1})
df_CR

,h,Ndof,error k=1,order k=1
0,0.306736,59,0.039923,0.000000
1,0.153368,279,0.011170,1.837575
2,0.076684,1127,0.002869,1.960808
3,0.038342,4455,0.000723,1.988908
4,0.019171,17639,0.000181,1.997224


In [72]:
from netgen.geom2d import SplineGeometry
from ngsolve import *
from ngsolve.webgui import Draw

# Construcción de una geometría 2D: Un cuadrado con un círculo restado en el centro
geo = SplineGeometry()
geo.AddRectangle((0, 0), (2, 2), bcs=["bottom", "right", "top", "left"], leftdomain=1)

# CORRECCIÓN: Se utiliza bc="hole" en lugar de bcs=["hole"]
geo.AddCircle(c=(1, 1), r=0.3, leftdomain=0, rightdomain=1, bc="hole")

# Generar malla adaptada a la geometría
mesh_hole = Mesh(geo.GenerateMesh(maxh=0.1))

# Espacio H1 con condiciones de Dirichlet en los bordes exteriores, libre en el agujero
Vh_hole = H1(mesh_hole, order=3, dirichlet="bottom|right|top|left")
u_h, v_h = Vh_hole.TnT()  # Trial and Test functions

# Fuente de calor asimétrica
f_heat = LinearForm(Vh_hole)
f_heat += 50 * exp(-10*((x-0.5)**2 + (y-0.5)**2)) * v_h * dx

a_heat = BilinearForm(Vh_hole)
a_heat += grad(u_h) * grad(v_h) * dx

a_heat.Assemble()
f_heat.Assemble()

gfu_heat = GridFunction(Vh_hole)
gfu_heat.vec.data = a_heat.mat.Inverse(Vh_hole.FreeDofs()) * f_heat.vec

# Visualización web
Draw(gfu_heat, mesh_hole, "Heat_Distribution")

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.24…

BaseWebGuiScene